# HW6 Colab runner — аблитерация + QLoRA-DPO

Ноутбук запускает полный `src.run_full` на бесплатном Colab T4: готовит сплиты, делает baseline-замер refusal на `Qwen/Qwen2.5-1.5B-Instruct`, аблитерирует refusal-направление, замеряет refusal после аблитерации, собирает DPO-пары, обучает QLoRA-DPO, замеряет refusal после DPO и упаковывает артефакты в zip для скачивания.

Перед запуском: `Runtime → Change runtime type → T4 GPU`. HF-токен не нужен — аблитерированная модель никуда не пушится, она воспроизводится из кода при том же seed (пайплайн идёт с `--skip-push`).

In [ ]:
import subprocess, sys
print('Python:', sys.version.split()[0])
print(subprocess.check_output(['nvidia-smi', '-L']).decode().strip())

In [ ]:
# Клонируем ветку hw_6 и идём в dz6/. Если репо уже есть — pull, чтобы
# подхватить свежие коммиты (пайплайн обновляется, cache-срыв сессии Colab
# не должен зафиксировать нас на устаревшем коде).
import os, subprocess, pathlib
REPO = 'https://github.com/dmagog/aith_DL_NLP.git'
BRANCH = 'hw_6'
WORK = pathlib.Path('/content/aith_DL_NLP')
if not WORK.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, str(WORK)], check=True)
else:
    subprocess.run(['git', '-C', str(WORK), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(WORK), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(WORK), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(WORK / 'dz6')
print('cwd:', os.getcwd())
print('head:', subprocess.check_output(['git', '-C', str(WORK), 'rev-parse', '--short', 'HEAD']).decode().strip())

In [ ]:
# Ставим зависимости из requirements.txt, чтобы версии совпадали с
# локальным прогоном. На Colab torch предустановлен, но мы upgrade'им
# accelerate/transformers/trl/peft под наш DPO-API.
import subprocess
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('installed')

In [ ]:
# Полный прогон. При обрыве — перезапуск этой ячейки продолжит с того
# места, где упал: все стадии идемпотентны и проверяют наличие артефактов.
#
# ВАЖНО: если после обновления кода пайплайна изменилась схема данных/сплитов,
# сбрось stale-кэш перед запуском — раскомментируй блок ниже.
import os, subprocess, sys, pathlib

# --- опциональный сброс кэша сплитов + baseline-метрик ---
# import shutil
# OUT = pathlib.Path('artifacts_hw6')
# for p in [OUT / 'sample', OUT / 'metrics_pretrained.json',
#           OUT / 'eval_pretrained_harmful.json', OUT / 'eval_pretrained_harmless.json']:
#     if p.exists():
#         shutil.rmtree(p) if p.is_dir() else p.unlink()
#         print('wiped', p)

pathlib.Path('artifacts_hw6').mkdir(parents=True, exist_ok=True)
log_path = pathlib.Path('artifacts_hw6/run.log')

env = {
    **os.environ,
    'PYTHONIOENCODING': 'utf-8',
    'PYTHONUTF8': '1',
    'PYTHONUNBUFFERED': '1',
    'LC_ALL': 'C.UTF-8',
    'LANG': 'C.UTF-8',
}
cmd = [
    sys.executable, '-u', '-m', 'src.run_full',
    '--out', 'artifacts_hw6',
    '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--skip-push',
    '--seed', '42',
]

# Читаем БАЙТАМИ по мере поступления — tqdm шлёт `\r`, а большие куски
# могут идти без `\n`; построчное чтение глушит весь прогресс. `read1`
# возвращает то, что уже пришло, без ожидания полного буфера.
with open(log_path, 'wb') as logf:
    proc = subprocess.Popen(
        cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=0,
    )
    try:
        while True:
            chunk = proc.stdout.read1(1024)
            if not chunk:
                break
            s = chunk.decode('utf-8', errors='replace')
            sys.stdout.write(s)
            sys.stdout.flush()
            logf.write(chunk)
            logf.flush()
    finally:
        rc = proc.wait()

if rc != 0:
    print(f'\n=== run_full exited with code {rc} ===')
    print(f'full log: {log_path} ({log_path.stat().st_size} bytes)')
    print('--- last 80 lines of log ---')
    with open(log_path, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    for line in lines[-80:]:
        print(line, end='')
    raise SystemExit(rc)

In [ ]:
# Проверяем, что все ключевые артефакты есть. Fail-fast: если какого-то
# файла нет — значит соответствующая стадия упала и надо разбираться.
import json, pathlib
OUT = pathlib.Path('artifacts_hw6')
required = [
    'sample/splits_info.json',
    'metrics_pretrained.json',
    'abliteration_info.json',
    'refusal_direction.pt',
    'metrics_abliterated.json',
    'dpo_data_info.json',
    'dpo_train_pairs.json',
    'dpo_val_pairs.json',
    'dpo_train_metrics.json',
    'dpo_log_history.json',
    'metrics_dpo.json',
    'run_summary.json',
]
missing = [p for p in required if not (OUT / p).exists()]
if missing:
    raise SystemExit(f'missing artifacts: {missing}')
print(json.dumps(json.loads((OUT / 'run_summary.json').read_text()), ensure_ascii=False, indent=2))

In [ ]:
# Упаковываем артефакты на скачивание. В zip НЕ включаем:
#   - abliterated_model/ — полные fp16-веса (~3 ГБ), в репо не нужны,
#     аблитерация воспроизводится из src/abliterate.py при том же seed;
#   - trainer/           — промежуточные checkpoint'ы DPO (gitignored);
#   - logs/              — машинный лог прогона (gitignored).
import shutil, pathlib, subprocess
stage = pathlib.Path('hw6_artifacts_to_download')
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir(parents=True)
for p in pathlib.Path('artifacts_hw6').rglob('*'):
    rel = p.relative_to('artifacts_hw6')
    if rel.parts and rel.parts[0] in ('abliterated_model', 'trainer', 'logs'):
        continue
    if p.is_dir():
        (stage / rel).mkdir(parents=True, exist_ok=True)
    else:
        (stage / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, stage / rel)
subprocess.run(['zip', '-rq', 'hw6_artifacts.zip', str(stage)], check=True)
print('packed:', pathlib.Path('hw6_artifacts.zip').stat().st_size / 1024 / 1024, 'MB')

In [ ]:
# Скачиваем zip к себе.
from google.colab import files  # type: ignore
files.download('hw6_artifacts.zip')

## Дальше

- Распаковать `hw6_artifacts.zip` в `dz6/artifacts_hw6/`, закоммитить (LoRA-адаптер ~70 МБ + JSON-ы, без полной модели).
- Открыть `dz6/hw6.ipynb` локально — он прочитает зафиксированные артефакты, GPU не нужен.